# Title: HRS Silver CDM DDL/DML Functional Specification
________________________________________
## 1. Document Information
|Document Name:|HRS Silver CDM DDL/DML Functional Specification|
| --- | --- |
|Version:|2.0|
|Author:|Perez|
|Last Updated:|2026-08-02|
|Target Runtime:|Databricks Runtime 15.x|
|SQL Dialect:|Spark SQL|
|Storage Format:|Delta Lake|
|Deployment Environment:|Development|
|AI Assistant:|Claude|
________________________________________

## 2. Objective

Use Claude to generate Databricks SQL DDL and DML scripts using Spark SQL compatible
with Databricks Runtime 15.x.

The generated SQL creates one Silver CDM table for a single RAND HRS survey section.

Return only SQL.

## 3. Purpose

This specification defines the physical implementation of one Silver CDM
survey-section table, and the load logic that populates it.

|Area | Description |
| --- | --- |
|Table Structure | Physical table definition|
|Relationships | Parent-child relationships|
|Column Definitions | Data types and nullability|
|Business Rules | Required ETL behavior|
|Mapping Rules | Source metadata to target columns|
|SQL Standards | Databricks DDL generation requirements|
|DML Standards | Databricks DML (load) generation requirements|

## 4. Table Parameters

Only this section changes between survey sections.

| Parameter | Value |
| --- | --- |
| SECTION_NAME | leave_behind_big5 |
| SECTION_DESCRIPTION | RAND HRS Codebook – Leave-Behind: Big 5 Personality Traits |
| CATALOG_NAME | dev_catalog |
| SCHEMA_NAME | slv_cdm_hrs |
| TABLE_NAME | hrs_leave_behind |
| FULL_TABLE_NAME | dev_catalog.slv_cdm_hrs.hrs_leave_behind |
| STORAGE_FORMAT | DELTA |
| TABLE_TYPE | Managed Table |
| LOAD_PATTERN | Insert Only |
| PRIMARY_KEY | hrs_leave_behind_id |

## 4a. Source Profile

| Property | Value | Notes |
| --- | --- | --- |
| SOURCE_TABLE_NAME | dev_catalog.brz_raw_hrs.randhrs1992_2022v1 | Same core wide file used for Demographics and Health |
| SOURCE_FILE_FAMILY | Core Longitudinal File (Leave-Behind items merged in) | Confirmed: Leave-Behind Big 5 variables are columns within the same wide source table, not a separate bronze table |
| SOURCE_ROW_GRAIN | One row per respondent | Same shape as Demographics/Health — requires UNPIVOT |
| SOURCE_NAMING_CONVENTION | `R{wave}LB{TRAIT}` for respondent variant (e.g. `R8LBNEUR`); spouse variant `S{wave}LB{TRAIT}` exists in source but is explicitly OUT OF SCOPE for this table per business decision below | Confirmed against codebook. One exception: Conscientiousness's source variable is `R{wave}LBCON5` (retains the "5 Sub-items" suffix) — this is confirmed correct, not a typo |
| SAMPLE_COVERAGE | Full eligible Leave-Behind sample, Waves 8–16 only | Big 5 items were not fielded before Wave 8. Source table has NO columns at all for `R1LBNEUR`...`R7LBNEUR` (and equivalents for the other 4 traits) — this is a column-existence gap, not a NULL-value gap |
| KNOWN_STRUCTURAL_DIFFERENCES | Wave coverage starts at Wave 8, not Wave 1. Spouse (S-prefix) variant excluded from this table by business decision (see §9). No multi-item aggregation required — each of the 5 traits is already a single pre-computed composite score in the source (Conscientiousness's `LBCON5` name reflects that it was built from 5 sub-items upstream, but the source column itself is a single scalar). |

## 5. Parent Tables

| Parent Table | Primary Key | Join Variable |
| --- | --- | --- |
| hrs_respondent | respondent_id | HHIDPN |
| hrs_wave | wave_id | wave_number |

## 6. Transformation Parameters

| RAND Variable Type | Parameter | Transformation |
| --- | --- | --- |
| CONT | CONT | TRY_CAST(<source_column> AS DECIMAL(10,2)) AS <target_column> |
| CATEG | CATEG | TRY_CAST(<source_column> AS <Databricks Type specified per-column in Section 12>) AS <target_column> |
| CHAR | CHAR | TRY_CAST(<source_column> AS STRING) AS <target_column> |

All 5 Big 5 target columns are CONT type — no CATEG or CHAR columns in this section.

## 7. Dependencies

| Object | Requirement |
| --- | --- |
| Catalog | dev_catalog |
| Schema | dev_catalog.slv_cdm_hrs |
| Parent Table | dev_catalog.slv_cdm_hrs.hrs_respondent |
| Parent Table | dev_catalog.slv_cdm_hrs.hrs_wave |
| Source Table | dev_catalog.brz_raw_hrs.randhrs1992_2022v1 |

## Parent Entities

| Parent Table | Primary Key |
| --- | --- |
| hrs_respondent | respondent_id |
| hrs_wave | wave_id |

## Child Entity

### Table

- hrs_leave_behind

## Relationships
| Parent | Child | Cardinality |
| --- | --- | --- |
| hrs_respondent | hrs_leave_behind | One to Many |
| hrs_wave | hrs_leave_behind | One to Many |

## Business Key
### Columns
- respondent_id
- wave_id

Together these columns uniquely identify one Big 5 personality observation.

## 9. Business Rules

| Rule | Description |
| --- | --- |
| Row Grain | One row per respondent per survey wave. **Waves 1–16 (full study span)** — chosen deliberately to keep grain consistent across every Silver CDM section (Demographics, Health, Big 5, and future sections), even though Big 5 items were only fielded from Wave 8 onward. Waves 1–7 will carry NULL values across all 5 business columns for every respondent. |
| Respondent FK | Required |
| Wave FK | Required |
| Duplicate Records | Not Allowed |
| Logical Business Key | respondent_id + wave_id |
| Unresolvable FK Handling | Silently exclude via INNER JOIN (consistent with Demographics/Health) |
| Spouse Data | Out of scope. Source contains `S{wave}LB{TRAIT}` spouse-variant columns; only the `R{wave}LB{TRAIT}` respondent variant is loaded into this table, per business decision. A future `hrs_leave_behind_spouse` table could be templated separately if spouse-level analysis is required. |

## 10. Physical Table Definition

| Property | Value |
| --- | --- |
| Storage Format | DELTA |
| Table Type | Managed Table |
| Table Comment | Stores RAND HRS Leave-Behind Big 5 Personality Trait observations |

## 11. Column Definitions

### Identity Column

| Column | Type | Nullable | Description |
| --- | --- | --- | --- |
| hrs_leave_behind_id | BIGINT | No | System-generated surrogate key |

Generated Always As Identity

### Foreign Keys

| Column | Type | Nullable | References |
| --- | --- | --- | --- |
| respondent_id | BIGINT | No | hrs_respondent.respondent_id |
| wave_id | BIGINT | No | hrs_wave.wave_id |

### Audit Columns

| Column | Type | Nullable | Description |
| --- | --- | --- | --- |
| create_date | DATE | No | Record creation date |
| update_date | DATE | No | Last update date |
| active | BOOLEAN | No | Active indicator |

### Identifier Columns

|Column | Type | Nullable | Description |
| --- | --- | --- | --- |
|hhidpn | int | Yes | Household Respondent Identifier |
|wave_number | string | No | Wave Number |

### Business Columns

The following columns are generated from the Source-to-Target Mapping Matrix (§12).

## 12. Source-to-Target Mapping Matrix

The Wave column's values are strings and must be handled as STRING type
throughout DDL and DML, consistent with Health. All 5 target columns below are
wave-varying (see §12a) and are populated only for Waves 8–16; Waves 1–7 are
populated with NULL via the scaffold pattern described in §13a of the template.

### Target Column: lbneur
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 8 | R8LBNEUR | R8LBNEUR: W8 Big 5 Neuroticism | Cont | lbneur | DECIMAL(10,2) | CONT | yes |
| 9 | R9LBNEUR | R9LBNEUR: W9 Big 5 Neuroticism | Cont | lbneur | DECIMAL(10,2) | CONT | yes |
| 10 | R10LBNEUR | R10LBNEUR: W10 Big 5 Neuroticism | Cont | lbneur | DECIMAL(10,2) | CONT | yes |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | R16LBNEUR | R16LBNEUR: W16 Big 5 Neuroticism | Cont | lbneur | DECIMAL(10,2) | CONT | yes |

### Target Column: lbext
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 8 | R8LBEXT | R8LBEXT: W8 Big 5 Extroversion | Cont | lbext | DECIMAL(10,2) | CONT | yes |
| 9 | R9LBEXT | R9LBEXT: W9 Big 5 Extroversion | Cont | lbext | DECIMAL(10,2) | CONT | yes |
| 10 | R10LBEXT | R10LBEXT: W10 Big 5 Extroversion | Cont | lbext | DECIMAL(10,2) | CONT | yes |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | R16LBEXT | R16LBEXT: W16 Big 5 Extroversion | Cont | lbext | DECIMAL(10,2) | CONT | yes |

### Target Column: lbopen
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 8 | R8LBOPEN | R8LBOPEN: W8 Big 5 Openness to experience | Cont | lbopen | DECIMAL(10,2) | CONT | yes |
| 9 | R9LBOPEN | R9LBOPEN: W9 Big 5 Openness to experience | Cont | lbopen | DECIMAL(10,2) | CONT | yes |
| 10 | R10LBOPEN | R10LBOPEN: W10 Big 5 Openness to experience | Cont | lbopen | DECIMAL(10,2) | CONT | yes |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | R16LBOPEN | R16LBOPEN: W16 Big 5 Openness to experience | Cont | lbopen | DECIMAL(10,2) | CONT | yes |

### Target Column: lbagr
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 8 | R8LBAGR | R8LBAGR: W8 Big 5 Agreeableness | Cont | lbagr | DECIMAL(10,2) | CONT | yes |
| 9 | R9LBAGR | R9LBAGR: W9 Big 5 Agreeableness | Cont | lbagr | DECIMAL(10,2) | CONT | yes |
| 10 | R10LBAGR | R10LBAGR: W10 Big 5 Agreeableness | Cont | lbagr | DECIMAL(10,2) | CONT | yes |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | R16LBAGR | R16LBAGR: W16 Big 5 Agreeableness | Cont | lbagr | DECIMAL(10,2) | CONT | yes |

### Target Column: lbcon5
| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation | Nullable |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 8 | R8LBCON5 | R8LBCON5: W8 Big 5 Conscientiousness 5 Sub-items | Cont | lbcon5 | DECIMAL(10,2) | CONT | yes |
| 9 | R9LBCON5 | R9LBCON5: W9 Big 5 Conscientiousness 5 Sub-items | Cont | lbcon5 | DECIMAL(10,2) | CONT | yes |
| 10 | R10LBCON5 | R10LBCON5: W10 Big 5 Conscientiousness 5 Sub-items | Cont | lbcon5 | DECIMAL(10,2) | CONT | yes |
| ... | ... | ... | ... | ... | ... | ... | ... |
| 16 | R16LBCON5 | R16LBCON5: W16 Big 5 Conscientiousness 5 Sub-items | Cont | lbcon5 | DECIMAL(10,2) | CONT | yes |

## 12a. Column Variance Declaration

| Target Column | Variance | Source Pattern | Notes |
| --- | --- | --- | --- |
| lbneur | Wave-varying | `R{w}LBNEUR`, w = 8..16 | No columns exist for w = 1..7 |
| lbext | Wave-varying | `R{w}LBEXT`, w = 8..16 | No columns exist for w = 1..7 |
| lbopen | Wave-varying | `R{w}LBOPEN`, w = 8..16 | No columns exist for w = 1..7 |
| lbagr | Wave-varying | `R{w}LBAGR`, w = 8..16 | No columns exist for w = 1..7 |
| lbcon5 | Wave-varying | `R{w}LBCON5`, w = 8..16 | No columns exist for w = 1..7; note "5" suffix retained in both source and target name |

All 5 business columns are wave-varying; none are wave-invariant. Unlike Health,
this section's wave-varying columns only have real source data for a subset of
the 1–16 wave universe — see §13a's Missing Source Columns rule.

## 13. SQL Generation Requirements (DDL)

| Requirement | Required |
| --- | --- |
| DROP TABLE IF EXISTS | Yes |
| CREATE TABLE | Yes |
| USING DELTA | Yes |
| GENERATED ALWAYS AS IDENTITY | Yes |
| Table Comment | Yes |
| Column Comments | Yes |
| Explicit Column Definitions | Yes |
| Uppercase SQL Keywords | Yes |
| Consistent Indentation | Yes |

## 13a. DML Generation Requirements

| Requirement | Rule |
| --- | --- |
| Unpivot Method | Multi-value `UNPIVOT ... INCLUDE NULLS` over the 5 business columns, applied ONLY to Waves 8–16 (the waves with real source columns) |
| Missing Source Columns for Waves 1–7 | Source table has no `R1LB*`...`R7LB*` columns at all. Rows for Waves 1–7 are produced via a synthetic scaffold: literal wave-number values `1`...`7` cross-joined against every distinct `HHIDPN` in the source, with `lbneur`, `lbext`, `lbopen`, `lbagr`, `lbcon5` all hardcoded `NULL`. Combined with the Wave 8–16 UNPIVOT result via `UNION ALL`. |
| Spouse Variant | `S{w}LB{TRAIT}` columns exist in source but are NOT selected into `source_base` — out of scope per §9 |
| NULL Handling | No `WHERE ... IS NOT NULL` filtering on business columns — consistent with Health |
| FK Join Type | `INNER JOIN` against `hrs_respondent` and `hrs_wave` |
| Type Casting | `TRY_CAST(... AS DECIMAL(10,2))` for all 5 columns per §6/§12 |
| wave_number Join Key | STRING on both sides — UNPIVOT aliases and scaffold literals both cast to STRING |
| Audit Column Population | `CURRENT_DATE()` for create_date/update_date; `active = TRUE` |

## 14. Constraint Requirements

### Primary Key Constraint
- CONSTRAINT pk_hrs_leave_behind PRIMARY KEY (hrs_leave_behind_id)

### Foreign Key Constraints
CONSTRAINT fk_hrs_leave_behind_respondent
    FOREIGN KEY (respondent_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_respondent (respondent_id),

CONSTRAINT fk_hrs_leave_behind_wave
    FOREIGN KEY (wave_id)
    REFERENCES dev_catalog.slv_cdm_hrs.hrs_wave (wave_id)

### Business Key Uniqueness Constraint

Databricks Runtime 15.x supports: PRIMARY KEY, FOREIGN KEY, UNIQUE, CHECK,
NOT NULL, GENERATED ALWAYS AS IDENTITY.

| Constraint Type | Required | Notes |
| --- | --- | --- |
| PRIMARY KEY | ✓ | Enforced at write time |
| FOREIGN KEY | ✓ | Enforced at write time |
| UNIQUE | ✓ | Required for business key |
| NOT NULL | ✓ | Required for audit + FK columns |
| CHECK | Optional | Could enforce 1.0–4.0 domain bounds on all 5 traits; not required |
| IDENTITY | ✓ | Required for surrogate key |

CONSTRAINT uq_hrs_leave_behind_respondent_wave UNIQUE (respondent_id, wave_id)

### Do not generate (DDL or DML)

Unsupported Objects:
- UPDATE, DELETE, MERGE
- Views, Indexes, Partitions, ZORDER, OPTIMIZE

(INSERT is required for DML deliverables.)

## Column Constraint Rules

### Identity Column
hrs_leave_behind_id BIGINT GENERATED ALWAYS AS IDENTITY
    CONSTRAINT pk_hrs_leave_behind PRIMARY KEY

### Foreign Keys
respondent_id BIGINT NOT NULL
    CONSTRAINT fk_hrs_leave_behind_respondent
        FOREIGN KEY REFERENCES dev_catalog.slv_cdm_hrs.hrs_respondent (respondent_id),

wave_id BIGINT NOT NULL
    CONSTRAINT fk_hrs_leave_behind_wave
        FOREIGN KEY REFERENCES dev_catalog.slv_cdm_hrs.hrs_wave (wave_id)

### Business Key
CONSTRAINT uq_hrs_leave_behind_respondent_wave UNIQUE (respondent_id, wave_id)

## 15. Validation Requirements

| Validation | Required |
| --- | --- |
| Table Exists | ✓ |
| Correct Schema | ✓ |
| Delta Format | ✓ |
| Managed Table | ✓ |
| Identity Column Exists | ✓ |
| Respondent FK Exists | ✓ |
| Wave FK Exists | ✓ |
| Audit Columns Exist | ✓ |
| respondent_id exists in hrs_respondent | ✓ |
| wave_id exists in hrs_wave | ✓ |
| No duplicate respondent_id + wave_id | ✓ |
| Rows loaded > 0 | ✓ |
| wave_number values within expected range (1–16) | ✓ |
| All 5 traits NULL for Waves 1–7 (confirms scaffold behaved as designed, not a data-quality defect) | ✓ |
| All 5 traits within 1.0–4.0 domain bounds where non-NULL (Waves 8–16) | Recommended |

## 16. Deliverables

|Item | Value|
|---|---|
|DDL File| /sql/ddl/create_hrs_leave_behind.sql|
|DML File| /sql/dml/insert_hrs_leave_behind.sql|
|Validation File| /sql/validation/validate_hrs_leave_behind.sql|
|Output| SQL Only|